In [1]:
import xarray as xr

In [2]:
ds1=xr.open_dataset(r"C:\Users\GFD LAB\Documents\Anwesha Chakraborty\data\era5_wndspd,rh,temp\9f9be389b356a5797f1cbf17b99fc417.nc")

In [4]:
ds1

<xarray.Dataset> Size: 1MB
Dimensions:     (valid_time: 2880, latitude: 5, longitude: 5)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 23kB 2025-01-01 ... 2025-04-30T23...
    expver      (valid_time) <U4 46kB ...
  * latitude    (latitude) float64 40B 29.0 28.75 28.5 28.25 28.0
  * longitude   (longitude) float64 40B 76.5 76.75 77.0 77.25 77.5
    number      int64 8B ...
Data variables:
    u10         (valid_time, latitude, longitude) float32 288kB ...
    v10         (valid_time, latitude, longitude) float32 288kB ...
    d2m         (valid_time, latitude, longitude) float32 288kB ...
    t2m         (valid_time, latitude, longitude) float32 288kB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-08-03T05:37 GRIB to CDM+CF via cfgrib-0.9.1...

In [5]:
import os
import glob
import numpy as np
import pandas as pd

In [6]:
folder=r"C:\Users\GFD LAB\Documents\Anwesha Chakraborty\data\era5_wndspd,rh,temp"

In [7]:
files = sorted(glob.glob(os.path.join(folder, "*.nc")))
print("Files found:", len(files))

Files found: 7


In [8]:
all_data = []

In [9]:
for file in files:
    ds = xr.open_dataset(file)
    # Delhi average
    t2m = ds["t2m"].mean(dim=["latitude", "longitude"])
    d2m = ds["d2m"].mean(dim=["latitude", "longitude"])
    u10 = ds["u10"].mean(dim=["latitude", "longitude"])
    v10 = ds["v10"].mean(dim=["latitude", "longitude"])
    # Kelvin to Celsius
    temp = t2m - 273.15
    dew = d2m - 273.15
    # Wind speed (m/s)
    wind = np.sqrt(u10**2 + v10**2)
    rh = 100 * (np.exp((17.625 * dew) / (243.04 + dew))/np.exp((17.625 * temp) / (243.04 + temp)))
    df = pd.DataFrame({
        "Datetime": ds["valid_time"].values,
        "Temperature": temp.values,
        "RH": rh.values,
        "WindSpeed": wind.values
    })

    all_data.append(df)
    ds.close()

In [10]:
era5 = pd.concat(all_data, ignore_index=True)

In [11]:
era5 = era5.drop_duplicates(subset="Datetime")

In [12]:
era5 = era5.sort_values("Datetime").reset_index(drop=True)

In [13]:
era5.shape

(21888, 4)

In [14]:
print("Start:", era5["Datetime"].min())
print("End:", era5["Datetime"].max())

Start: 2024-01-01 00:00:00
End: 2026-06-30 23:00:00


In [15]:
era5.to_csv(
    r"C:\Users\GFD LAB\Documents\Anwesha Chakraborty\output\ERA5_Meteorology.csv",index=False)

In [16]:
era5.describe().T

,count,mean,min,25%,50%,75%,max,std
Datetime,21888,2025-03-31 23:30:00.000000256,2024-01-01 00:00:00,2024-08-15 23:45:00,2025-03-31 23:30:00,2025-11-14 23:15:00,2026-06-30 23:00:00,NaN
Temperature,21888.0,24.988245,5.025818,18.60701,26.879349,31.038818,46.171387,8.468093
RH,21888.0,65.839943,10.570574,47.890717,69.864651,84.651287,100.001022,21.815382
WindSpeed,21888.0,2.281851,0.010084,1.501211,2.120533,2.854183,8.333353,1.156407
